# RandLANet Classification Debug with Pytorch Geometric

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
import time
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [5]:
from torch_pointcloud.models.randlanet import LocalSpatialEncoding as LocalSpatialEncoding_TP
from torch_pointcloud.ops import knn, knn_interpolate
from torch_pointcloud import _C

## Open3D Blocks

In [6]:
class SharedMLP(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=1,
        stride=1,
        transpose=False,
        bn=True,
        activation_fn=None,
    ):
        super(SharedMLP, self).__init__()

        if transpose:
            self.conv = nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=(kernel_size - 1) // 2,
            )
        else:
            self.conv = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=(kernel_size - 1) // 2,
            )

        self.batch_norm = (
            nn.BatchNorm2d(out_channels, eps=1e-6, momentum=0.01) if bn else None
        )
        self.activation_fn = activation_fn

    def forward(self, input):
        """Forward pass of the Module.

        Args:
            input: torch.Tensor of shape (B, dim_in, N, K)

        Returns:
            torch.Tensor, shape (B, dim_out, N, K)

        """
        x = self.conv(input)
        if self.batch_norm:
            x = self.batch_norm(x)
        if self.activation_fn:
            x = self.activation_fn(x)
        return x


class LocalSpatialEncoding(nn.Module):
    def __init__(self, dim_in, dim_out, num_neighbors, encode_pos=False):
        super(LocalSpatialEncoding, self).__init__()

        self.num_neighbors = num_neighbors
        self.mlp = SharedMLP(dim_in, dim_out, activation_fn=nn.LeakyReLU(0.2))
        self.encode_pos = encode_pos

    def gather_neighbor(self, coords, neighbor_indices):
        """Gather features based on neighbor indices.

        Args:
            coords: torch.Tensor of shape (B, N, d)
            neighbor_indices: torch.Tensor of shape (B, N, K)

        Returns:
            gathered neighbors of shape (B, dim, N, K)

        """
        B, N, K = neighbor_indices.size()
        dim = coords.shape[2]

        extended_indices = neighbor_indices.unsqueeze(1).expand(B, dim, N, K)
        extended_coords = coords.transpose(-2, -1).unsqueeze(-1).expand(B, dim, N, K)
        neighbor_coords = torch.gather(
            extended_coords, 2, extended_indices
        )  # (B, dim, N, K)

        return neighbor_coords

    def forward(self, coords, features, neighbor_indices, relative_features=None):
        """Forward pass of the Module.

        Args:
            coords: coordinates of the pointcloud
                torch.Tensor of shape (B, N, 3)
            features: features of the pointcloud.
                torch.Tensor of shape (B, d, N, 1)
            neighbor_indices: indices of k neighbours.
                torch.Tensor of shape (B, N, K)
            relative_features: relative neighbor features calculated
              on first pass. Required for second pass.

        Returns:
            torch.Tensor of shape (B, 2*d, N, K)

        """
        # finding neighboring points
        B, N, K = neighbor_indices.size()

        if self.encode_pos:
            neighbor_coords = self.gather_neighbor(coords, neighbor_indices)

            extended_coords = coords.transpose(-2, -1).unsqueeze(-1).expand(B, 3, N, K)

            relative_pos = extended_coords - neighbor_coords
            relative_dist = torch.sqrt(
                torch.sum(torch.square(relative_pos), dim=1, keepdim=True)
            )

            relative_features = torch.cat(
                [relative_dist, relative_pos, extended_coords, neighbor_coords], dim=1
            )

        else:
            if relative_features is None:
                raise ValueError(
                    "LocalSpatialEncoding: Require relative_features for second pass."
                )

        relative_features = self.mlp(relative_features)

        neighbor_features = self.gather_neighbor(
            features.transpose(1, 2).squeeze(3), neighbor_indices
        )

        return (
            torch.cat([neighbor_features, relative_features], dim=1),
            relative_features,
        )


class AttentivePooling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(AttentivePooling, self).__init__()

        self.score_fn = nn.Sequential(
            nn.Linear(in_channels, in_channels), nn.Softmax(dim=-2)
        )
        self.mlp = SharedMLP(in_channels, out_channels, activation_fn=nn.LeakyReLU(0.2))

    def forward(self, x):
        """Forward pass of the Module.

        Args:
            x: torch.Tensor of shape (B, dim_in, N, K).

        Returns:
            torch.Tensor of shape (B, d_out, N, 1).

        """
        # computing attention scores
        scores = self.score_fn(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

        # sum over the neighbors
        features = torch.sum(scores * x, dim=-1, keepdim=True)  # shape (B, d_in, N, 1)

        return self.mlp(features)


class LocalFeatureAggregation(nn.Module):
    def __init__(self, d_in, d_out, num_neighbors):
        super(LocalFeatureAggregation, self).__init__()

        self.num_neighbors = num_neighbors

        self.mlp1 = SharedMLP(d_in, d_out // 2, activation_fn=nn.LeakyReLU(0.2))
        self.lse1 = LocalSpatialEncoding(10, d_out // 2, num_neighbors, encode_pos=True)
        self.pool1 = AttentivePooling(d_out, d_out // 2)

        self.lse2 = LocalSpatialEncoding(d_out // 2, d_out // 2, num_neighbors)
        self.pool2 = AttentivePooling(d_out, d_out)
        self.mlp2 = SharedMLP(d_out, 2 * d_out)

        self.shortcut = SharedMLP(d_in, 2 * d_out)
        self.lrelu = nn.LeakyReLU()

    def forward(self, coords, feat, neighbor_indices):
        """Forward pass of the Module.

        Args:
            coords: coordinates of the pointcloud
                torch.Tensor of shape (B, N, 3).
            feat: features of the pointcloud.
                torch.Tensor of shape (B, d, N, 1)
            neighbor_indices: Indices of neighbors.

        Returns:
            torch.Tensor of shape (B, 2*d_out, N, 1).

        """
        # _, neighbor_indices = knn(coords, coords, k=16)
        x = self.mlp1(feat)
        print(f"[in lse1] {coords = }")
        print(f"[in lse1] {x = }")
        print(f"[in lse1] {neighbor_indices = }")
        x, neighbor_features = self.lse1(coords, x, neighbor_indices)
        print(f"[out lse1] {x = }")
        print(f"[out lse1] {neighbor_features = }")
        x = self.pool1(x)
        x, _ = self.lse2(
            coords, x, neighbor_indices, relative_features=neighbor_features
        )
        x = self.pool2(x)

        return self.lrelu(self.mlp2(x) + self.shortcut(feat))

## Torch PointCloud Implementation

### Test LocalSpatialEncoding

In [7]:
from torch_pointcloud.models.randlanet import LocalSpatialEncoding as LocalSpatialEncoding_TP
from torch_pointcloud.ops import knn, knn_interpolate

In [8]:
coords = torch.rand(32, 1024, 3)
features = torch.rand(32, 3, 1024).unsqueeze(-1)
neighbor_dists, neighbor_idxs = knn(coords, coords, k=16)

In [9]:
# Original implementation
mlp1 = SharedMLP(3, 8, activation_fn=nn.LeakyReLU(0.2))
lse1 = LocalSpatialEncoding(10, 8, num_neighbors=16, encode_pos=True)
pool1 = AttentivePooling(16, 8)
lse2 = LocalSpatialEncoding(8, 8, num_neighbors=16)
pool2 = AttentivePooling(16, 16)

x = mlp1(features)
x, relative_features = lse1(coords, x, neighbor_idxs)
x = pool1(x)
x, _ = lse2(coords, x, neighbor_idxs, relative_features=relative_features)
x = pool2(x)

In [10]:
# Custom implementation
mlp1 = SharedMLP(3, 8, activation_fn=nn.LeakyReLU(0.2))
lse1 = LocalSpatialEncoding_TP(10, 8, num_neighbors=16, encode_coords=True)
pool1 = AttentivePooling(16, 8)
lse2 = LocalSpatialEncoding_TP(8, 8, num_neighbors=16)
pool2 = AttentivePooling(16, 16)

x = mlp1(features)
x, relative_features = lse1(coords, x, neighbor_idxs)
x = pool1(x)
x, _ = lse2(coords, x, neighbor_idxs, relative_features=relative_features)
x = pool2(x)

### Test LocalFeatureAggregation

In [11]:
from torch_pointcloud.models.randlanet import LocalFeatureAggregation as LocalFeatureAggregation_TP
from torch_pointcloud.ops import knn, knn_interpolate


coords = torch.rand(32, 1024, 3).cuda()
features = torch.rand(32, 3, 1024).unsqueeze(-1).cuda()
neighbor_dists, neighbor_idxs = knn(coords, coords, k=16)

In [12]:
lfa = LocalFeatureAggregation(3, 16, num_neighbors=16)
lfa.cuda()

out_original = lfa(coords, features, neighbor_idxs)
print(f"{out_original.shape = }")

[in lse1] coords = tensor([[[0.5428, 0.7461, 0.9859],
         [0.4135, 0.1573, 0.1938],
         [0.1660, 0.6418, 0.2007],
         ...,
         [0.5000, 0.2157, 0.0206],
         [0.0610, 0.6979, 0.8891],
         [0.4386, 0.6500, 0.3729]],

        [[0.5702, 0.7376, 0.2433],
         [0.8017, 0.6433, 0.5796],
         [0.7185, 0.5970, 0.3665],
         ...,
         [0.3134, 0.0389, 0.6199],
         [0.7369, 0.5810, 0.4535],
         [0.6022, 0.4824, 0.8175]],

        [[0.7780, 0.7576, 0.7303],
         [0.6862, 0.6284, 0.5540],
         [0.5316, 0.5579, 0.1267],
         ...,
         [0.5519, 0.6958, 0.7236],
         [0.7588, 0.2506, 0.4301],
         [0.9599, 0.8055, 0.0687]],

        ...,

        [[0.7288, 0.6246, 0.7974],
         [0.8278, 0.4057, 0.5629],
         [0.9921, 0.2090, 0.8620],
         ...,
         [0.6495, 0.5172, 0.8757],
         [0.4115, 0.9911, 0.8103],
         [0.4544, 0.7234, 0.5273]],

        [[0.2685, 0.4890, 0.5771],
         [0.0258, 0.9608, 0.

In [13]:
lfa_tp = LocalFeatureAggregation_TP(3, 16, num_neighbors=16)
lfa_tp.cuda()

out_tp = lfa_tp(coords, features.squeeze(-1))
print(f"{out_tp.shape = }")

out_tp.shape = torch.Size([32, 32, 1024])


In [14]:
from torch_pointcloud.models.randlanet import RandLANetSegmentation


model = RandLANetSegmentation(in_channels=3, num_classes=10, num_neighbors=16).cuda()

torch.random.manual_seed(0)
coords = torch.rand((1, 10_000, 3)).cuda()
features = None
model(coords, features)

tensor([[[ 0.5037,  0.4527,  0.6287,  ...,  1.3349,  0.7884,  0.8722],
         [ 0.0280,  0.0055,  0.1721,  ...,  0.0620, -0.0935, -0.7966],
         [-0.4503, -0.2494,  0.2505,  ..., -0.2802, -0.6570, -0.2259],
         ...,
         [-0.1052, -0.2602, -0.6998,  ..., -1.7816, -0.6235, -0.7382],
         [ 0.6820, -0.4294, -0.0175,  ...,  0.2548, -0.2854,  0.4031],
         [ 0.0168,  0.0723, -0.5769,  ..., -0.5635, -1.0795, -0.7747]]],
       device='cuda:0', grad_fn=<SqueezeBackward1>)

In [243]:
state_dict_mapping = {
    'mlp1.conv.weight': 'mlp1.conv.weight',
    'mlp1.conv.bias': 'mlp1.conv.bias',
    'mlp1.batch_norm.weight': 'mlp1.batch_norm.weight',
    'mlp1.batch_norm.bias': 'mlp1.batch_norm.bias',
    'mlp1.batch_norm.running_mean': 'mlp1.batch_norm.running_mean',
    'mlp1.batch_norm.running_var': 'mlp1.batch_norm.running_var',
    'mlp1.batch_norm.num_batches_tracked': 'mlp1.batch_norm.num_batches_tracked',
    'mlp2.conv.weight': 'mlp2.conv.weight',
    'mlp2.conv.bias': 'mlp2.conv.bias',
    'mlp2.batch_norm.weight': 'mlp2.batch_norm.weight',
    'mlp2.batch_norm.bias': 'mlp2.batch_norm.bias',
    'mlp2.batch_norm.running_mean': 'mlp2.batch_norm.running_mean',
    'mlp2.batch_norm.running_var': 'mlp2.batch_norm.running_var',
    'mlp2.batch_norm.num_batches_tracked': 'mlp2.batch_norm.num_batches_tracked',
    'shortcut.conv.weight': 'mlp_skip.conv.weight',
    'shortcut.conv.bias': 'mlp_skip.conv.bias',
    'shortcut.batch_norm.weight': 'mlp_skip.batch_norm.weight',
    'shortcut.batch_norm.bias': 'mlp_skip.batch_norm.bias',
    'shortcut.batch_norm.running_mean': 'mlp_skip.batch_norm.running_mean',
    'shortcut.batch_norm.running_var': 'mlp_skip.batch_norm.running_var',
    'shortcut.batch_norm.num_batches_tracked': 'mlp_skip.batch_norm.num_batches_tracked',
    'lse1.mlp.conv.weight': 'lse1.mlp.conv.weight',
    'lse1.mlp.conv.bias': 'lse1.mlp.conv.bias',
    'lse1.mlp.batch_norm.weight': 'lse1.mlp.batch_norm.weight',
    'lse1.mlp.batch_norm.bias': 'lse1.mlp.batch_norm.bias',
    'lse1.mlp.batch_norm.running_mean': 'lse1.mlp.batch_norm.running_mean',
    'lse1.mlp.batch_norm.running_var': 'lse1.mlp.batch_norm.running_var',
    'lse1.mlp.batch_norm.num_batches_tracked': 'lse1.mlp.batch_norm.num_batches_tracked',
    'lse2.mlp.conv.weight': 'lse2.mlp.conv.weight',
    'lse2.mlp.conv.bias': 'lse2.mlp.conv.bias',
    'lse2.mlp.batch_norm.weight': 'lse2.mlp.batch_norm.weight',
    'lse2.mlp.batch_norm.bias': 'lse2.mlp.batch_norm.bias',
    'lse2.mlp.batch_norm.running_mean': 'lse2.mlp.batch_norm.running_mean',
    'lse2.mlp.batch_norm.running_var': 'lse2.mlp.batch_norm.running_var',
    'lse2.mlp.batch_norm.num_batches_tracked': 'lse2.mlp.batch_norm.num_batches_tracked',
    'pool1.score_fn.0.weight': 'pool1.attn.0.weight',
    'pool1.score_fn.0.bias': 'pool1.attn.0.bias',
    'pool1.mlp.conv.weight': 'pool1.mlp.conv.weight',
    'pool1.mlp.conv.bias': 'pool1.mlp.conv.bias',
    'pool1.mlp.batch_norm.weight': 'pool1.mlp.batch_norm.weight',
    'pool1.mlp.batch_norm.bias': 'pool1.mlp.batch_norm.bias',
    'pool1.mlp.batch_norm.running_mean': 'pool1.mlp.batch_norm.running_mean',
    'pool1.mlp.batch_norm.running_var': 'pool1.mlp.batch_norm.running_var',
    'pool1.mlp.batch_norm.num_batches_tracked': 'pool1.mlp.batch_norm.num_batches_tracked',
    'pool2.score_fn.0.weight': 'pool2.attn.0.weight',
    'pool2.score_fn.0.bias': 'pool2.attn.0.bias',
    'pool2.mlp.conv.weight': 'pool2.mlp.conv.weight',
    'pool2.mlp.conv.bias': 'pool2.mlp.conv.bias',
    'pool2.mlp.batch_norm.weight': 'pool2.mlp.batch_norm.weight',
    'pool2.mlp.batch_norm.bias': 'pool2.mlp.batch_norm.bias',
    'pool2.mlp.batch_norm.running_mean': 'pool2.mlp.batch_norm.running_mean',
    'pool2.mlp.batch_norm.running_var': 'pool2.mlp.batch_norm.running_var',
    'pool2.mlp.batch_norm.num_batches_tracked': 'pool2.mlp.batch_norm.num_batches_tracked',
}


lfa_state_dict = lfa.state_dict()
lfa_tp_state_dict = lfa_tp.state_dict()

for key, value in lfa_state_dict.items():
    tp_key = state_dict_mapping.get(key)
    assert lfa_tp_state_dict[tp_key].shape == lfa_state_dict[key].shape, f"{tp_key = }, {key = }"
    lfa_tp_state_dict[tp_key] = lfa_state_dict[key]

lfa_tp.load_state_dict(lfa_tp_state_dict)

<All keys matched successfully>

In [248]:
torch.allclose(out_original1, out_original2)

True

In [246]:
out_original1 = lfa(coords, features, neighbor_idxs)
out_tp = lfa_tp(coords, features)

torch.allclose(out_original1, out_original2, atol=1e-5)

[in lse1] coords = tensor([[[0.5142, 0.1771, 0.1046],
         [0.4113, 0.2374, 0.4736],
         [0.7829, 0.3972, 0.4622],
         ...,
         [0.6202, 0.9435, 0.9158],
         [0.5416, 0.9985, 0.8570],
         [0.9825, 0.4239, 0.2853]],

        [[0.6663, 0.2837, 0.2029],
         [0.3402, 0.9411, 0.8968],
         [0.5238, 0.1772, 0.8921],
         ...,
         [0.5371, 0.6432, 0.3620],
         [0.6416, 0.9161, 0.6724],
         [0.1812, 0.5008, 0.2269]],

        [[0.4641, 0.7448, 0.0782],
         [0.5633, 0.6201, 0.8669],
         [0.2934, 0.4426, 0.1521],
         ...,
         [0.2035, 0.5637, 0.7093],
         [0.6234, 0.2371, 0.9990],
         [0.2932, 0.9927, 0.5038]],

        ...,

        [[0.0323, 0.7516, 0.4973],
         [0.0015, 0.3029, 0.2418],
         [0.6135, 0.1173, 0.8718],
         ...,
         [0.4639, 0.0106, 0.9424],
         [0.8803, 0.4908, 0.2562],
         [0.3764, 0.5143, 0.1022]],

        [[0.2450, 0.7714, 0.6466],
         [0.7559, 0.1827, 0.

True

In [230]:
torch.allclose(out_original, out_tp, atol=1e-6)

False

In [4]:
import torch
import torch.nn as nn


m = nn.LogSoftmax(dim=1)
loss = nn.NLLLoss()
# input is of size N x C = 3 x 5
input = torch.randn(3, 5, requires_grad=True)
# each element in target has to have 0 <= value < C
target = torch.tensor([1, 0, 4])
pred = m(input)

print(f"{pred.shape = }")
print(f"{target.shape = }")

output = loss(pred, target)
output.backward()

pred.shape = torch.Size([3, 5])
target.shape = torch.Size([3])


In [5]:
# 2D loss example (used, for example, with image inputs)
N, C = 5, 4
loss = nn.NLLLoss()
# input is of size N x C x height x width
data = torch.randn(N, 16, 10, 10)
conv = nn.Conv2d(16, C, (3, 3))
m = nn.LogSoftmax(dim=1)
# each element in target has to have 0 <= value < C
target = torch.empty(N, 8, 8, dtype=torch.long).random_(0, C)
pred = m(conv(data))

print(f"{pred.shape = }")
print(f"{target.shape = }")

output = loss(pred, target)
output.backward()

pred.shape = torch.Size([5, 4, 8, 8])
target.shape = torch.Size([5, 8, 8])


In [7]:
# 1 - target.shape = torch.Size([32, 4096])
# 1 - seg_pred.shape = torch.Size([32, 13, 4096])
# 2 - target.shape = torch.Size([32, 4096])
# 2 - seg_pred.shape = torch.Size([32, 4096, 13])
# 3 - target.shape = torch.Size([32, 4096])
# 3 - seg_pred.shape = torch.Size([131072, 13])
# 4 - batch_label.shape = (131072,)
# 5 - target.shape = torch.Size([131072])
# 5 - seg_pred.shape = torch.Size([131072, 13])